[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Testing and Packaging](https://johnfisher-ai.github.io/Python-Visual-Guides/testing-and-packaging.html)

# Test Structure &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell sets up the project's folders and `run_pytest`, and the cells after it write the
module and two of the test files the notebook's worked examples left, which the tasks use. Run them
first, then the tasks in order, since tasks 2 to 4 build on the file task 1 writes. The last cell
removes the scratch folder.


In [1]:
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

PROJECT = Path("scratch/stations")
(PROJECT / "tests").mkdir(parents=True, exist_ok=True)
os.environ["NO_COLOR"] = "1"                  # programs started from here print without color codes
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"   # and keep no compiled copies, which a quick rewrite can outrun


def pytest_report(*arguments, folder=PROJECT):
    """What python -m pytest prints when it runs in the folder, less what differs between computers."""
    settings = {"COLUMNS": "80", "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1", "PYTHONNODEBUGRANGES": "1"}
    finished = subprocess.run([sys.executable, "-m", "pytest", "--no-header", *arguments],
                              cwd=folder, capture_output=True, text=True, env={**os.environ, **settings})
    report = finished.stdout + finished.stderr
    report = report.replace(f"{Path(folder).resolve()}/", "")          # the folder's own path
    report = re.sub(r"\S*/_pytest/", "_pytest/", report)                # the path to pytest's own files
    return re.sub(r" in \d+\.\d+s\b", "", report).rstrip()              # the time the run took


def run_pytest(*arguments, folder=PROJECT):
    """Run python -m pytest in the folder, as a terminal would, and print its report."""
    print(pytest_report(*arguments, folder=folder))


print("ready:", PROJECT)


ready: scratch/stations


In [2]:
%%writefile scratch/stations/readings.py
"""Readings from the weather stations, and each station's mean temperature."""

import statistics


def parse_reading(line):
    """A (station, celsius) pair from a line such as 'Bergen,4.2'. An empty reading is None.

    A minus sign written as U+2212, as some spreadsheets write it, reads as a hyphen-minus.
    """
    station, celsius = line.strip().split(",")
    celsius = celsius.replace("\u2212", "-")
    return station, float(celsius) if celsius else None


def mean(values):
    """The mean of the readings that are not None, or None when there are none."""
    present = [value for value in values if value is not None]
    return statistics.fmean(present) if present else None


def summarize(lines):
    """Each station's mean temperature, from lines of readings. A blank line is skipped."""
    by_station = {}
    for line in lines:
        if not line.strip():
            continue
        station, celsius = parse_reading(line)
        by_station.setdefault(station, []).append(celsius)
    return {station: mean(values) for station, values in by_station.items()}


def to_fahrenheit(celsius):
    """A temperature in degrees Celsius, in degrees Fahrenheit."""
    return celsius * 9 / 5 + 32


Writing scratch/stations/readings.py


In [3]:
%%writefile scratch/stations/tests/test_mean.py
from readings import mean


class TestMean:
    def test_of_two_readings(self):
        assert mean([4.2, 5.8]) == 5.0

    def test_skips_missing_readings(self):
        assert mean([4.2, None, 5.8]) == 5.0

    def test_of_no_readings_is_none(self):
        assert mean([None, None]) is None

    def test_of_one_reading_is_that_reading(self):
        assert mean([-6.3]) == -6.3


Writing scratch/stations/tests/test_mean.py


In [4]:
%%writefile scratch/stations/tests/test_summary.py
from readings import summarize


def lines_for(station, *readings):
    """Lines of readings from one station, with None for an empty reading."""
    return [f"{station},{'' if reading is None else reading}" for reading in readings]


def test_a_station_with_only_empty_readings_has_no_mean():
    lines = lines_for("Bergen", 4.2) + lines_for("Svalbard", None, None)

    summary = summarize(lines)

    assert summary["Svalbard"] is None


Writing scratch/stations/tests/test_summary.py


**1.** A test in three steps.


In [5]:
%%writefile scratch/stations/tests/test_tasks.py
from readings import to_fahrenheit


def test_minus_forty_is_the_same_in_both_scales():
    celsius = -40

    fahrenheit = to_fahrenheit(celsius)

    assert fahrenheit == -40


Writing scratch/stations/tests/test_tasks.py


In [6]:
run_pytest("tests/test_tasks.py", "-v")


============================= test session starts ==============================
collecting ... collected 1 item

tests/test_tasks.py::test_minus_forty_is_the_same_in_both_scales PASSED  [100%]

============================== 1 passed ===============================


-40 is the one temperature that reads the same in both scales, which the name says, so a failure
would say what went wrong without the file open.


**2.** One test of three behaviors, as three tests.


In [7]:
%%writefile scratch/stations/tests/test_tasks.py
from readings import mean, to_fahrenheit


def test_minus_forty_is_the_same_in_both_scales():
    celsius = -40

    fahrenheit = to_fahrenheit(celsius)

    assert fahrenheit == -40


def test_mean_of_two_readings():
    assert mean([4.2, 5.8]) == 5.0


def test_mean_of_a_missing_reading_is_none():
    assert mean([None]) is None


def test_mean_of_one_reading_is_that_reading():
    assert mean([-6.3]) == -6.3


Overwriting scratch/stations/tests/test_tasks.py


In [8]:
run_pytest("tests/test_tasks.py", "-v")


============================= test session starts ==============================
collecting ... collected 4 items

tests/test_tasks.py::test_minus_forty_is_the_same_in_both_scales PASSED  [ 25%]
tests/test_tasks.py::test_mean_of_two_readings PASSED                    [ 50%]
tests/test_tasks.py::test_mean_of_a_missing_reading_is_none PASSED       [ 75%]
tests/test_tasks.py::test_mean_of_one_reading_is_that_reading PASSED     [100%]

============================== 4 passed ===============================


Each test has one `assert` here, which is common, although one behavior sometimes needs more.


**3.** The tests of task 2, in a class.


In [9]:
%%writefile scratch/stations/tests/test_tasks.py
from readings import mean, to_fahrenheit


def test_minus_forty_is_the_same_in_both_scales():
    celsius = -40

    fahrenheit = to_fahrenheit(celsius)

    assert fahrenheit == -40


class TestMeanTasks:
    def test_of_two_readings(self):
        assert mean([4.2, 5.8]) == 5.0

    def test_of_a_missing_reading_is_none(self):
        assert mean([None]) is None

    def test_of_one_reading_is_that_reading(self):
        assert mean([-6.3]) == -6.3


Overwriting scratch/stations/tests/test_tasks.py


In [10]:
run_pytest("-k", "TestMeanTasks", "-v")


============================= test session starts ==============================
collecting ... collected 9 items / 6 deselected / 3 selected

tests/test_tasks.py::TestMeanTasks::test_of_two_readings PASSED          [ 33%]
tests/test_tasks.py::TestMeanTasks::test_of_a_missing_reading_is_none PASSED [ 66%]
tests/test_tasks.py::TestMeanTasks::test_of_one_reading_is_that_reading PASSED [100%]

======================= 3 passed, 6 deselected ========================


The names inside the class drop `mean`, since the class's name carries it. `-k TestMeanTasks` matched
the class and so all three of its tests, and none of `TestMean`'s, whose name does not contain
`TestMeanTasks`.


**4.** One test in the class, by its node ID.


In [11]:
run_pytest("tests/test_tasks.py::TestMeanTasks::test_of_a_missing_reading_is_none", "-v")


============================= test session starts ==============================
collecting ... collected 1 item

tests/test_tasks.py::TestMeanTasks::test_of_a_missing_reading_is_none PASSED [100%]

============================== 1 passed ===============================


The file, the class and the method, joined by `::`.


**5.** Two tests that depend on each other, made independent.


In [12]:
%%writefile scratch/stations/tests/test_order.py
STATIONS = []


def test_adding_a_station():
    STATIONS.append("Alta")
    assert STATIONS == ["Alta"]


def test_the_first_station_is_alta():
    assert STATIONS[0] == "Alta"


Writing scratch/stations/tests/test_order.py


In [13]:
run_pytest("tests/test_order.py", "-q")
print()
report = pytest_report("tests/test_order.py::test_the_first_station_is_alta", "-q")
print("\n".join(line for line in report.splitlines() if line.startswith(("E ", "FAILED", "1 failed"))))


..                                                                       [100%]
2 passed

E       IndexError: list index out of range
FAILED tests/test_order.py::test_the_first_station_is_alta - IndexError: list...
1 failed


In [14]:
%%writefile scratch/stations/tests/test_order.py
def test_adding_a_station():
    stations = []

    stations.append("Alta")

    assert stations == ["Alta"]


def test_the_first_station_is_alta():
    stations = ["Alta", "Bergen"]

    first = stations[0]

    assert first == "Alta"


Overwriting scratch/stations/tests/test_order.py


In [15]:
run_pytest("tests/test_order.py", "-q")
print()
run_pytest("tests/test_order.py::test_the_first_station_is_alta", "-q")


..                                                                       [100%]
2 passed

.                                                                        [100%]
1 passed


Alone, the second test found `STATIONS` empty and raised `IndexError`: it passed together with the
first test only because that test filled the list. Now each test arranges its own list, and passes in
any run.


**6.** A helper for two stations.


In [16]:
%%writefile scratch/stations/tests/test_two_stations.py
from readings import summarize


def lines_for(station, *readings):
    """Lines of readings from one station, with None for an empty reading."""
    return [f"{station},{'' if reading is None else reading}" for reading in readings]


def test_two_stations_with_one_reading_each():
    lines = lines_for("Bergen", 4.2) + lines_for("Oslo", -2.4)

    summary = summarize(lines)

    assert summary == {"Bergen": 4.2, "Oslo": -2.4}


Writing scratch/stations/tests/test_two_stations.py


In [17]:
run_pytest("tests/test_two_stations.py", "-v")


============================= test session starts ==============================
collecting ... collected 1 item

tests/test_two_stations.py::test_two_stations_with_one_reading_each PASSED [100%]

============================== 1 passed ===============================


The helper is copied from `tests/test_summary.py` rather than imported from it, since one test file
importing another is fragile. The **Fixtures** notebook shows how tests in several files share one
helper, through `conftest.py`.

Last, remove the scratch folder:


In [18]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Test Structure](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/testing-and-packaging/03-test-structure.ipynb)  &nbsp;&middot;&nbsp;  [Testing and Packaging Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/testing-and-packaging.html)
